# Water Quality Index (WQI) Forecasting

Forecasting module focusing on selected Classical, Machine Learning, Deep Learning, and Multivariate models.

In [34]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import json
import warnings
from IPython.display import display
warnings.filterwarnings('ignore')

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import MinMaxScaler
import xgboost as xgb
from prophet import Prophet
import torch
import torch.nn as nn
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.api import VAR

## 1. Data Loading & Preprocessing

In [35]:
# Load Dataset
df = pd.read_csv('dataset1_nonghan_water_quality.csv')

# Drop null dates and sort
df = df.dropna(subset=['date'])
df['date'] = pd.to_datetime(df['date'], errors='coerce')
df = df.dropna(subset=['date']).sort_values('date')

# We use Monthly frequency ('ME') for time series modeling
ts_data = df.groupby(pd.Grouper(key='date', freq='ME')).agg({
    'WQI_al_score': 'mean',
    'pH': 'mean',
    'DO_mg_L': 'mean',
    'BOD_mg_L': 'mean',
    'COD_mg_L': 'mean',
    'NH3_mg_L': 'mean'
}).reset_index()

# Forward fill missing values in resampled data
ts_data = ts_data.ffill().bfill()
print(f"Resampled Monthly Data Shape: {ts_data.shape}")
display(ts_data.head())

Resampled Monthly Data Shape: (24, 7)


,date,WQI_al_score,pH,DO_mg_L,BOD_mg_L,COD_mg_L,NH3_mg_L
0,2024-01-31,69.709569,7.142727,5.558325,3.270287,16.353110,0.685177
1,2024-02-29,69.284314,7.173039,5.546618,3.367549,15.678431,0.700289
2,2024-03-31,69.562745,7.148480,5.619216,3.237696,14.691176,0.677917
3,2024-04-30,67.515054,7.148710,5.368925,3.521989,15.922581,0.732989
4,2024-05-31,67.961644,7.141233,5.495342,3.450868,15.555251,0.743667


## 2. Evaluation Metrics Setup

In [36]:
def calculate_metrics(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100 if np.mean(y_true) != 0 else 0
    r2 = r2_score(y_true, y_pred)
    return {'MAE': round(mae, 3), 'RMSE': round(rmse, 3), 'MAPE': round(mape, 3), 'R2': round(r2, 3)}

results_table = []

## 3. Classical Models

### 3.1 SARIMA

In [37]:
train_size = int(len(ts_data) * 0.8)
train, test = ts_data['WQI_al_score'][:train_size], ts_data['WQI_al_score'][train_size:]

seasonal_order = (1, 1, 1, 12) if len(train) >= 24 else (0, 0, 0, 0)
model_sarima = SARIMAX(train, order=(1, 1, 1), seasonal_order=seasonal_order)
fit_sarima = model_sarima.fit(disp=False)
pred_sarima = fit_sarima.forecast(steps=len(test))

metrics_sarima = calculate_metrics(test, pred_sarima)
results_table.append({'Model': 'SARIMA', **metrics_sarima})
print("SARIMA:", metrics_sarima)

SARIMA: {'MAE': 3.778, 'RMSE': np.float64(5.026), 'MAPE': np.float64(5.603), 'R2': -1.17}


### 3.2 Holt-Winters (ETS)

In [38]:
seasonal_periods = 12 if len(train) >= 24 else None
seasonal = 'add' if seasonal_periods else None

model_ets = ExponentialSmoothing(train, trend='add', seasonal=seasonal, seasonal_periods=seasonal_periods)
fit_ets = model_ets.fit()
pred_ets = fit_ets.forecast(steps=len(test))

metrics_ets = calculate_metrics(test, pred_ets)
results_table.append({'Model': 'Holt-Winters', **metrics_ets})
print("Holt-Winters:", metrics_ets)

Holt-Winters: {'MAE': 3.135, 'RMSE': np.float64(4.628), 'MAPE': np.float64(4.698), 'R2': -0.84}


### 3.3 Prophet

In [39]:
df_prophet = ts_data[['date', 'WQI_al_score']].rename(columns={'date': 'ds', 'WQI_al_score': 'y'})
train_p, test_p = df_prophet.iloc[:train_size], df_prophet.iloc[train_size:]

m_prophet = Prophet(yearly_seasonality=True)
m_prophet.fit(train_p)

future = m_prophet.make_future_dataframe(periods=len(test_p), freq='ME')
forecast = m_prophet.predict(future)
pred_prophet = forecast['yhat'].iloc[-len(test_p):].values

metrics_prophet = calculate_metrics(test_p['y'].values, pred_prophet)
results_table.append({'Model': 'Prophet', **metrics_prophet})
print("Prophet:", metrics_prophet)

18:06:53 - cmdstanpy - INFO - Chain [1] start processing
18:07:03 - cmdstanpy - INFO - Chain [1] done processing


Prophet: {'MAE': 2.5, 'RMSE': np.float64(3.037), 'MAPE': np.float64(3.653), 'R2': 0.208}


## 4. Machine Learning Models

### 4.1 Feature Engineering for ML

In [40]:
df_ml = ts_data.copy()
df_ml['month'] = df_ml['date'].dt.month
df_ml['quarter'] = df_ml['date'].dt.quarter
df_ml['lag_1'] = df_ml['WQI_al_score'].shift(1)
df_ml['lag_2'] = df_ml['WQI_al_score'].shift(2)
df_ml['rolling_mean_3'] = df_ml['WQI_al_score'].rolling(3).mean()
df_ml['rolling_std_3'] = df_ml['WQI_al_score'].rolling(3).std()

df_ml = df_ml.dropna()
features = ['month', 'quarter', 'lag_1', 'lag_2', 'rolling_mean_3', 'rolling_std_3']
X = df_ml[features]
y = df_ml['WQI_al_score']

X_train_ml, X_test_ml = X.iloc[:int(len(X)*0.8)], X.iloc[int(len(X)*0.8):]
y_train_ml, y_test_ml = y.iloc[:int(len(X)*0.8)], y.iloc[int(len(X)*0.8):]

### 4.2 XGBoost

In [41]:
model_xgb = xgb.XGBRegressor(n_estimators=100, learning_rate=0.1, objective='reg:squarederror')
model_xgb.fit(X_train_ml, y_train_ml)
pred_xgb = model_xgb.predict(X_test_ml)

metrics_xgb = calculate_metrics(y_test_ml.values, pred_xgb)
results_table.append({'Model': 'XGBoost', **metrics_xgb})
print("XGBoost:", metrics_xgb)

XGBoost: {'MAE': 1.648, 'RMSE': np.float64(2.182), 'MAPE': np.float64(2.415), 'R2': 0.591}


## 5. Deep Learning (LSTM)

In [42]:
scaler = MinMaxScaler(feature_range=(-1, 1))
ts_scaled = scaler.fit_transform(ts_data[['WQI_al_score']].values)

def create_sequences(data, seq_length):
    xs, ys = [], []
    for i in range(len(data)-seq_length):
        xs.append(data[i:(i+seq_length)])
        ys.append(data[i+seq_length])
    return np.array(xs), np.array(ys)

seq_length = 6
X_lstm, y_lstm = create_sequences(ts_scaled, seq_length)
train_size_lstm = int(len(X_lstm) * 0.8)

X_train_t = torch.tensor(X_lstm[:train_size_lstm], dtype=torch.float32)
y_train_t = torch.tensor(y_lstm[:train_size_lstm], dtype=torch.float32)
X_test_t = torch.tensor(X_lstm[train_size_lstm:], dtype=torch.float32)
y_test_t = torch.tensor(y_lstm[train_size_lstm:], dtype=torch.float32)

class LSTMModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm = nn.LSTM(input_size=1, hidden_size=64, num_layers=1, batch_first=True)
        self.linear = nn.Linear(64, 1)
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.linear(out[:, -1, :])

model_lstm = LSTMModel()
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model_lstm.parameters(), lr=0.01)

epochs = 200
for epoch in range(epochs):
    optimizer.zero_grad()
    y_pred_t = model_lstm(X_train_t)
    loss = criterion(y_pred_t, y_train_t)
    loss.backward()
    optimizer.step()

model_lstm.eval()
with torch.no_grad():
    pred_lstm_scaled = model_lstm(X_test_t).numpy()
    
pred_lstm = scaler.inverse_transform(pred_lstm_scaled).flatten()
y_test_lstm_inv = scaler.inverse_transform(y_test_t.numpy()).flatten()

metrics_lstm = calculate_metrics(y_test_lstm_inv, pred_lstm)
results_table.append({'Model': 'LSTM', **metrics_lstm})
print("LSTM:", metrics_lstm)

LSTM: {'MAE': 1.895, 'RMSE': np.float64(2.163), 'MAPE': np.float32(2.697), 'R2': 0.611}


## 6. Multivariate (VAR)

In [43]:
var_data = ts_data[['WQI_al_score', 'DO_mg_L', 'BOD_mg_L', 'COD_mg_L', 'pH']].dropna()
train_var, test_var = var_data.iloc[:train_size], var_data.iloc[train_size:]

model_var = VAR(train_var)
fit_var = model_var.fit(maxlags=1)

lag_order = fit_var.k_ar
forecast_input = train_var.values[-lag_order:]
pred_var_raw = fit_var.forecast(y=forecast_input, steps=len(test_var))
pred_var = pred_var_raw[:, 0]

metrics_var = calculate_metrics(test_var['WQI_al_score'].values, pred_var)
results_table.append({'Model': 'VAR', **metrics_var})
print("VAR:", metrics_var)

VAR: {'MAE': 2.584, 'RMSE': np.float64(3.329), 'MAPE': np.float64(3.815), 'R2': 0.048}


## 7. Model Evaluation & Comparison

In [ ]:
res_df = pd.DataFrame(results_table).sort_values('RMSE')
best_model_name = res_df.iloc[0]['Model']
print(f"Best Model based on RMSE: {best_model_name}")

# Full future forecast using Prophet
m_prophet = Prophet(yearly_seasonality=True)
m_prophet.fit(df_prophet)
future = m_prophet.make_future_dataframe(periods=12, freq='ME')
forecast = m_prophet.predict(future)

hist_json = [{"date": str(d.date()), "value": round(v, 2)} for d, v in zip(df_prophet['ds'], df_prophet['y'])]
fcst_json = [{"date": str(d.date()), "value": round(y, 2), "lower": round(yl, 2), "upper": round(yu, 2)} 
             for d, y, yl, yu in zip(forecast['ds'].iloc[-12:], forecast['yhat'].iloc[-12:], forecast['yhat_lower'].iloc[-12:], forecast['yhat_upper'].iloc[-12:])]

output_json = {
    "model": best_model_name,
    "best_rmse": float(res_df.iloc[0]['RMSE']),
    "historical": hist_json,
    "forecast": fcst_json
}

print("JSON Output preview:")
print(json.dumps(output_json, indent=2, ensure_ascii=False)[:500] + "...\n}")

# Model Comparison Table
table_fig = go.Figure(data=[go.Table(
    header=dict(values=list(res_df.columns),
                fill_color='paleturquoise',
                align='left'),
    cells=dict(values=[res_df[col] for col in res_df.columns],
               fill_color='lavender',
               align='left'))
])
table_fig.update_layout(title="Model Comparison", margin=dict(t=40, l=0, r=0, b=0))
table_fig.show()

# Forecast Visualization
fig = go.Figure()
fig.add_trace(go.Scatter(x=df_prophet['ds'], y=df_prophet['y'], mode='lines+markers', name='Historical WQI'))
fig.add_trace(go.Scatter(x=forecast['ds'].iloc[-12:], y=forecast['yhat'].iloc[-12:], mode='lines+markers', name='Forecasted WQI'))
fig.add_trace(go.Scatter(x=forecast['ds'].iloc[-12:], y=forecast['yhat_upper'].iloc[-12:], mode='lines', line=dict(width=0), showlegend=False))
fig.add_trace(go.Scatter(x=forecast['ds'].iloc[-12:], y=forecast['yhat_lower'].iloc[-12:], mode='lines', fill='tonexty', fillcolor='rgba(255, 165, 0, 0.2)', line=dict(width=0), name='Confidence Interval'))

fig.update_layout(title="WQI Forecast (Next 12 Months)", xaxis_title="Date", yaxis_title="WQI Score", template="plotly_white")
fig.show()

## 8. Proper Backtesting Evaluation

In [48]:
import plotly.express as px
from plotly.subplots import make_subplots
import numpy as np
import pandas as pd
from sklearn.metrics import r2_score
from IPython.display import display

# 80/20 chronological split based on the FULL dataset
n_total = len(df_prophet)
n_train = int(n_total * 0.8)
train_backtest = df_prophet.iloc[:n_train]
test_backtest = df_prophet.iloc[n_train:]

print(f"Total periods: {n_total}")
print(f"Train periods: {len(train_backtest)}")
print(f"Test periods: {len(test_backtest)}")
print(f"Best Model selected for Backtesting: {best_model_name}")

pred_test = None
pred_lower = None
pred_upper = None

# We must ensure pred_test exactly matches len(test_backtest)
def align_predictions(preds, target_len, last_val):
    if len(preds) == target_len:
        return preds
    elif len(preds) > target_len:
        return preds[-target_len:]
    else:
        diff = target_len - len(preds)
        return np.concatenate((preds, np.full(diff, last_val)))

if best_model_name == 'Prophet':
    bt_model = Prophet(yearly_seasonality=True)
    bt_model.fit(train_backtest)
    future_bt = bt_model.make_future_dataframe(periods=len(test_backtest), freq='ME')
    forecast_bt = bt_model.predict(future_bt)
    
    pred_test = forecast_bt['yhat'].iloc[-len(test_backtest):].values
    pred_lower = forecast_bt['yhat_lower'].iloc[-len(test_backtest):].values
    pred_upper = forecast_bt['yhat_upper'].iloc[-len(test_backtest):].values
    
elif best_model_name == 'SARIMA':
    bt_model = SARIMAX(train_backtest['y'].values, order=(1, 1, 1), seasonal_order=(1, 1, 1, 12) if len(train_backtest) >= 24 else (0, 0, 0, 0))
    bt_fit = bt_model.fit(disp=False)
    fc = bt_fit.get_forecast(steps=len(test_backtest))
    pred_test = fc.predicted_mean
    pred_lower = fc.conf_int(alpha=0.05)[:, 0]
    pred_upper = fc.conf_int(alpha=0.05)[:, 1]
    
elif best_model_name == 'Holt-Winters':
    seasonal_periods = 12 if len(train_backtest) >= 24 else None
    seasonal = 'add' if seasonal_periods else None
    bt_model = ExponentialSmoothing(train_backtest['y'].values, trend='add', seasonal=seasonal, seasonal_periods=seasonal_periods)
    bt_fit = bt_model.fit()
    pred_test = bt_fit.forecast(steps=len(test_backtest))
    std = np.std(bt_fit.resid)
    pred_lower = pred_test - 1.96 * std
    pred_upper = pred_test + 1.96 * std
    
elif best_model_name == 'XGBoost':
    bt_model = xgb.XGBRegressor(n_estimators=100, learning_rate=0.1, objective='reg:squarederror')
    bt_model.fit(X_train_ml, y_train_ml)
    preds = bt_model.predict(X_test_ml)
    pred_test = align_predictions(preds, len(test_backtest), train_backtest['y'].values[-1])
    std = np.std(y_train_ml.values - bt_model.predict(X_train_ml))
    pred_lower = pred_test - 1.96 * std
    pred_upper = pred_test + 1.96 * std
    
elif best_model_name == 'LSTM':
    preds = pred_lstm
    pred_test = align_predictions(preds, len(test_backtest), train_backtest['y'].values[-1])
    
    # Calculate std from training set for CI
    pred_train_lstm = scaler.inverse_transform(model_lstm(X_train_t).detach().numpy()).flatten()
    y_train_lstm_inv = scaler.inverse_transform(y_train_t.numpy()).flatten()
    std = np.std(y_train_lstm_inv - pred_train_lstm)
    pred_lower = pred_test - 1.96 * std
    pred_upper = pred_test + 1.96 * std
    
elif best_model_name == 'VAR':
    preds = pred_var
    pred_test = align_predictions(preds, len(test_backtest), train_backtest['y'].values[-1])
    # Approximate std based on test errors for var since we don't have train residuals easily
    std = np.std(test_var['WQI_al_score'].values - pred_var)
    pred_lower = pred_test - 1.96 * std
    pred_upper = pred_test + 1.96 * std
    
else:
    # Fallback
    pred_test = np.full(len(test_backtest), train_backtest['y'].values[-1])
    pred_lower = pred_test - 5
    pred_upper = pred_test + 5

actual = test_backtest['y'].values
error = actual - pred_test
abs_error = np.abs(error)
pct_error = (abs_error / actual) * 100

compare_df = pd.DataFrame({
    'Date': test_backtest['ds'].dt.date,
    'Actual': actual,
    'Forecast': np.round(pred_test, 2),
    'Error': np.round(error, 2),
    'Absolute Error': np.round(abs_error, 2),
    'Percentage Error (%)': np.round(pct_error, 2)
})

print("\nTop 5 Largest Prediction Errors:")
display(compare_df.sort_values(by='Absolute Error', ascending=False).head(5))

def smape(A, F):
    return 100/len(A) * np.sum(2 * np.abs(F - A) / (np.abs(A) + np.abs(F)))

bt_metrics = {
    'MAE': np.mean(abs_error),
    'RMSE': np.sqrt(np.mean(error**2)),
    'MAPE': np.mean(pct_error),
    'SMAPE': smape(actual, pred_test),
    'R2': r2_score(actual, pred_test)
}

print("\n--- Backtesting Evaluation Metrics on Holdout Set ---")
for k, v in bt_metrics.items():
    print(f"{k}: {v:.3f}")

# 1. Actual vs Forecast Line Chart
fig1 = go.Figure()
fig1.add_trace(go.Scatter(x=train_backtest['ds'], y=train_backtest['y'], mode='lines+markers', name='Historical (Train)', marker=dict(color='blue')))
fig1.add_trace(go.Scatter(x=test_backtest['ds'], y=actual, mode='lines+markers', name='Actual (Test)', marker=dict(color='green')))

# Setup hover text for forecast line to include Date, Actual, Forecast, Error
hover_text = [f"Date: {d}<br>Actual: {a:.2f}<br>Forecast: {f:.2f}<br>Error: {e:.2f}" 
              for d, a, f, e in zip(test_backtest['ds'].dt.date, actual, pred_test, error)]

fig1.add_trace(go.Scatter(
    x=test_backtest['ds'], y=pred_test, 
    mode='lines+markers', name='Forecast', 
    marker=dict(color='red'),
    text=hover_text, hoverinfo="text"
))
fig1.add_trace(go.Scatter(x=test_backtest['ds'], y=pred_upper, mode='lines', line=dict(width=0), showlegend=False, hoverinfo='skip'))
fig1.add_trace(go.Scatter(x=test_backtest['ds'], y=pred_lower, mode='lines', fill='tonexty', fillcolor='rgba(255, 0, 0, 0.2)', line=dict(width=0), name='95% CI', hoverinfo='skip'))
fig1.update_layout(title="Backtesting: Actual vs Forecast (Holdout 20%)", xaxis_title="Date", yaxis_title="WQI Score", template="plotly_white", hovermode="closest")
fig1.show()

# 2. Residual Analysis Dashboard
fig2 = make_subplots(rows=2, cols=2, subplot_titles=("Residuals Plot", "Residual Histogram", "Actual vs Predicted", "Absolute Error over Time"))
fig2.add_trace(go.Scatter(x=test_backtest['ds'], y=error, mode='lines+markers', name='Residual', marker=dict(color='purple')), row=1, col=1)
fig2.add_hline(y=0, line_dash="dash", line_color="black", row=1, col=1)
fig2.add_trace(go.Histogram(x=error, name='Error Freq', nbinsx=10, marker=dict(color='orange')), row=1, col=2)
fig2.add_trace(go.Scatter(x=actual, y=pred_test, mode='markers', name='Actual vs Pred', marker=dict(color='teal')), row=2, col=1)
min_v = min(min(actual), min(pred_test))
max_v = max(max(actual), max(pred_test))
fig2.add_trace(go.Scatter(x=[min_v, max_v], y=[min_v, max_v], mode='lines', line=dict(dash='dash', color='gray'), name='Ideal'), row=2, col=1)
fig2.add_trace(go.Bar(x=test_backtest['ds'], y=abs_error, name='Abs Error', marker=dict(color='firebrick')), row=2, col=2)
fig2.update_layout(title_text="Backtesting Residual Analysis Dashboard", height=700, template="plotly_white")
fig2.show()

Total periods: 24
Train periods: 19
Test periods: 5
Best Model selected for Backtesting: LSTM

Top 5 Largest Prediction Errors:


,Date,Actual,Forecast,Error,Absolute Error,Percentage Error (%)
21,2025-10-31,72.646759,67.29,5.36,5.36,7.38
23,2025-12-31,68.173684,73.27,-5.10,5.10,7.47
20,2025-09-30,73.503352,69.93,3.57,3.57,4.86
22,2025-11-30,64.938119,68.28,-3.34,3.34,5.15
19,2025-08-31,73.370370,75.91,-2.54,2.54,3.46



--- Backtesting Evaluation Metrics on Holdout Set ---
MAE: 3.983
RMSE: 4.126
MAPE: 5.665
SMAPE: 5.654
R2: -0.462
